In [1]:
##########################################
## Python/OpenRouter code to perform USMLE Med Question Answer
##
## Dataset comes from: https://github.com/chat-data-llc/medical_chat_performance_evaluation/blob/main/test_datasets/USMLE/Medical%20Chat%20USMLE%20Correctness%20Check%20-%20Test%201.csv
##
## Author: Christopher Meaney
## Date: June 2026
##########################################

In [2]:
##########
## Package dependencies
##########

In [3]:
## For connection to open router
from openai import OpenAI

## For connection to Open Router, CSV output, etc.
import os

## For dataframes and data wrangling
import pandas as pd
import numpy as np

## For timing
import time

## For JSON data structure
import json

## For regular expressions --- possibly needed to fix broken JSON structures returned by LLM
import re

In [4]:
## Pandas options
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 100)

In [5]:
##########
## File paths for input/output data
##########
WORK_DIR = r"/path/to/project/"

INPUT_EXAMS_CSV = os.path.join(
    WORK_DIR,
    "data",
    "Medical Chat USMLE Correctness Check - Test 1_CM.csv"
)

OUTPUT_DIR = os.path.join(WORK_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Checkpoint / run output
OUTPUT_CHECKPOINT_CSV = os.path.join(
    OUTPUT_DIR,
    "model_results.csv"
)

## Summary outputs
PERFORMANCE_SUMMARY_CSV = os.path.join(OUTPUT_DIR, "performance_summary.csv")

In [6]:
# help(pd.read_csv)

In [7]:
##########
## Import title abstract data
##########

In [8]:
df = pd.read_csv(INPUT_EXAMS_CSV, 
                 encoding="utf-8",
                 encoding_errors ="ignore"
                )
df.head(n=6)

,Number,TypeQuestion,Question,CorrectResponse,ChatGPT_Output
0,1.1,MC-NJ,Serum LDL-cholesterol concentrations are maeasured in blood samples collected from 25 healthy vo...,(B) 120-140,"With a 95% confidence level, the true mean for the population from which this sample was drawn f..."
1,1.2,MC-J,Serum LDL-cholesterol concentrations are maeasured in blood samples collected from 25 healthy vo...,(B) 120-140,The standard error of the mean is the standard deviation of the sampling distribution of the mea...
2,1.3,OE,Serum LDL-cholesterol concentrations are maeasured in blood samples collected from 25 healthy vo...,(B) 120-140,The standard error of the mean is a measure of the variability of the sample mean. It is calcula...
3,2.1,MC-NJ,"A 48-year-old man dies suddenly of a cardiac arrhythmia. Six weeks ago, he was resuscitated from...",(E) Normal kidney,It is difficult to accurately answer this question without more information. Oliguric renal fail...
4,2.2,MC-J,"A 48-year-old man dies suddenly of a cardiac arrhythmia. Six weeks ago, he was resuscitated from...",(E) Normal kidney,It is difficult to say for certain what the examination of the patient's kidney at autopsy would...
5,2.3,OE,"A 48-year-old man dies suddenly of a cardiac arrhythmia. Six weeks ago, he was resuscitated from...",(E) Normal kidney,It is impossible for me to say with certainty what the result of the examination of the patient'...


In [9]:
## Dimensions of title/abstract corpus
df.shape

(271, 5)

In [10]:
## Names of columns
pd.Series(df.columns)

0             Number
1       TypeQuestion
2           Question
3    CorrectResponse
4     ChatGPT_Output
dtype: str

In [11]:
## Number of words (assuming white space tokenization) from USMLE/MCQ corpus
df["question_word_count"] = df["Question"].astype(str).str.split().str.len()

df[["question_word_count"]].describe()

,question_word_count
count,271.000000
mean,118.726937
std,46.518009
min,41.000000
25%,87.000000
50%,111.000000
75%,141.000000
max,285.000000


In [12]:
df[["question_word_count"]].quantile([0, 0.01, 0.025, 0.05, 0.1, 0.25, 0.50, 0.75, 0.90, 0.95, 0.975, 0.99, 1])

,question_word_count
0.000,41.00
0.010,42.00
0.025,49.75
0.050,56.00
0.100,66.00
0.250,87.00
0.500,111.00
0.750,141.00
0.900,184.00
0.950,206.50


In [13]:
## Type question
df.TypeQuestion.value_counts()

TypeQuestion
MC-NJ    94
MC-J     94
OE       83
Name: count, dtype: int64

In [14]:
df_ = df[df['TypeQuestion']=='MC-NJ']
df_.shape

(94, 6)

In [15]:
## Correct Response
df_["CorrectResponse_R"] = df_["CorrectResponse"].str.split().str[0].str.strip("()")
df_.CorrectResponse_R.value_counts().sort_index()

CorrectResponse_R
A    21
B    18
C    14
D    24
E    13
F     3
G     1
Name: count, dtype: int64

In [16]:
##########
## Randomly split/divide USMLE datset
##
## 1) 10 question "resevoir" used to select examples for prompting LLM
## 2) 84 question test dataset
##
## Note: We perform this split so no USMLE shot/example questions are ever used in test set evaluation
##########

In [17]:
SEED = 12345
rng = np.random.default_rng(SEED)

In [18]:
## df_ is your 94-row MC-NJ dataframe
df_pool = df_.copy().reset_index(drop=True)

## Choose 10 reservoir questions
reservoir_idx = rng.choice(df_pool.index, size=10, replace=False)

## Split
df_reservoir = df_pool.loc[reservoir_idx].copy()
df_test = df_pool.drop(reservoir_idx).copy()

[df_reservoir.shape, df_test.shape]

[(10, 7), (84, 7)]

In [19]:
# reservoir_idx

In [20]:
##########
## Connection to OpenRouter
########## 

In [21]:
client = OpenAI(
    api_key=os.environ.get("OPENROUTER_API_KEY"), ## Need to bash: export OPENROUTER_API_KEY=[your_key]
    base_url="https://openrouter.ai/api/v1",      ## OpenRouter model endpoints
)

print("Client ready")

Client ready


In [22]:
###################################
##
## Prompt engineering, components:
##
## - Role:                          (None, Medical Expert)
## - Explanation / reasoning cue:   (None, Brief explanation in JSON with constraints)
## - Examples / shots:              (None/zero, one-shot, few-shot)
##
## 2*2*3 ==> 12 prompt templates to explore impacts of role, explanation, shots ==> under factorial design
##
###################################

In [23]:
##
## Helper function to select example/shots for inclusion in prompt
##
def make_shot_example(row):
    q = str(row["Question"]).strip()
    ans = str(row["CorrectResponse_R"]).strip()

    payload = {"answer": ans}

    payload_str = json.dumps(payload, ensure_ascii=False)

    template = """
Question:
<QUESTION>

Answer:
<PAYLOAD>
""".strip()

    return (
        template
        .replace("<QUESTION>", q)
        .replace("<PAYLOAD>", payload_str)
    )

In [24]:
# make_shot_example(row=df_reservoir.iloc[0])

In [25]:
##
## Helper function to sample shots from resevoir
##
def sample_shots(df_reservoir, n_shots, rng=None):

    sampled = df_reservoir.sample(
        n=n_shots,
        replace=False,
        random_state=rng,
    )

    shot_strings = [
        make_shot_example(row)
        for _, row in sampled.iterrows()
    ]

    return "\n\n".join(shot_strings)

In [26]:
# sample_shots(df_reservoir, n_shots=3, random_state=SEED)

In [27]:
##
## Define 12 types of prompts to expect in this experiment
##
PROMPTS = [
    {"name": "base_zero_noex",  "role": False, "explanation": False, "shots": 0},
    {"name": "base_one_noex",   "role": False, "explanation": False, "shots": 1},
    {"name": "base_few_noex",   "role": False, "explanation": False, "shots": 3},
    {"name": "base_zero_expl",  "role": False, "explanation": True,  "shots": 0},
    {"name": "base_one_expl",   "role": False, "explanation": True,  "shots": 1},
    {"name": "base_few_expl",   "role": False, "explanation": True,  "shots": 3},
    {"name": "role_zero_noex",  "role": True,  "explanation": False, "shots": 0},
    {"name": "role_one_noex",   "role": True,  "explanation": False, "shots": 1},
    {"name": "role_few_noex",   "role": True,  "explanation": False, "shots": 3},
    {"name": "role_zero_expl",  "role": True,  "explanation": True,  "shots": 0},
    {"name": "role_one_expl",   "role": True,  "explanation": True,  "shots": 1},
    {"name": "role_few_expl",   "role": True,  "explanation": True,  "shots": 3},
]

In [28]:
#####################
## Function to build prompt, based on MCQ, prompt specification (see above), shots/examples (from reservoir), and RNG
#####################
def build_messages(question, spec, df_reservoir, rng):

    if spec["role"]:
        system_prompt = """
You are a medical expert answering USMLE-style multiple-choice questions.
Return only valid JSON.
""".strip()
    else:
        system_prompt = "Return only valid JSON."

    if spec["explanation"]:
        json_schema = '{"answer":"<A|B|C|D|E|F|G>","explanation":"<one short sentence>"}'
        rules_block = """
Rules:
- "answer" must be exactly one uppercase letter from A, B, C, D, E, F, or G.
- "explanation" must be exactly one sentence and no more than 25 words.
- Do not include any text outside the JSON object.
""".strip()
    else:
        json_schema = '{"answer":"<A|B|C|D|E|F|G>"}'
        rules_block = """
Rules:
- "answer" must be exactly one uppercase letter from A, B, C, D, E, F, or G.
- Do not include any text outside the JSON object.
""".strip()

    shots_block = ""
    if spec["shots"] > 0:
        sampled_shots = sample_shots(
            df_reservoir=df_reservoir,
            n_shots=spec["shots"],
            rng=rng
        )
        shots_block = f"""
Here are worked examples.

{sampled_shots}

Now answer the next question.
""".strip()

    user_prompt = f"""
Answer the question by selecting one option from A, B, C, D, E, F, or G.

Return exactly this JSON schema:
{json_schema}

{rules_block}

{shots_block}

Question:
{question}
""".strip()

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

In [29]:
##
## Loop over prompt specs (defined above) and investigate what the resulting fixed prompts might look like
##

In [30]:
test_question = str(df_test.iloc[0]["Question"])

rng = np.random.default_rng(54321)

for spec in PROMPTS:
    print("=" * 80)
    print("PROMPT:", spec["name"])
    print("ROLE:", spec["role"], "EXPLANATION:", spec["explanation"], "SHOTS:", spec["shots"])

    messages = build_messages(
        question=test_question,
        spec=spec,
        df_reservoir=df_reservoir,
        rng=rng,
    )

    print("\nSYSTEM:")
    print(messages[0]["content"])

    print("\nUSER:")
    print(messages[1]["content"])
    print("\n")

PROMPT: base_zero_noex
ROLE: False EXPLANATION: False SHOTS: 0

SYSTEM:
Return only valid JSON.

USER:
Answer the question by selecting one option from A, B, C, D, E, F, or G.

Return exactly this JSON schema:
{"answer":"<A|B|C|D|E|F|G>"}

Rules:
- "answer" must be exactly one uppercase letter from A, B, C, D, E, F, or G.
- Do not include any text outside the JSON object.



Question:
Serum LDL-cholesterol concentrations are maeasured in blood samples collected from 25 healthy volunteers. The data follow a normal distribution. The mean and standard deviation for this group are 130 mg/dL and 25 mg/dL, respectively. The standard error of the mean is 5.0. With a 95% confidence level, the true mean for the population from which this sample was drawn falls within which of the following ranges (in mg/dL)?

(A) 105-155
(B) 120-140
(C) 125-135
(D) 128-132
(E) 129-131


PROMPT: base_one_noex
ROLE: False EXPLANATION: False SHOTS: 1

SYSTEM:
Return only valid JSON.

USER:
Answer the question by s

In [31]:
###########
## Specify the LLM you want to answer USMLE questions
###########

In [32]:
MODELS = [
    #"nvidia/nemotron-3-nano-30b-a3b:free",
    "deepseek/deepseek-v4-flash",
    #"openai/gpt-4o-mini",
    #"anthropic/claude-3-haiku",
    #"google/gemini-2.5-flash",
    #"x-ai/grok-4.20"
    #"openai/gpt-5.3-chat",
    #"anthropic/claude-opus-4.7",
    #"google/gemini-3.5-flash",
    #"x-ai/grok-4.3",
    
]

In [33]:
#MODEL_SPECS = [
#    {
#        "model": "nvidia/nemotron-3-nano-30b-a3b:free",
#        "use_system": True,
#    }
#]

MODEL_SPECS = [
    {
        "model": "deepseek/deepseek-v4-flash",
        "use_system": True,
    }
]

#
#MODEL_SPECS = [
#    {
#        "model": "openai/gpt-4o-mini",
#        "use_system": True,
#    },
#    {
#        "model": "anthropic/claude-3-haiku",
#        "use_system": True,
#    },
#    {
#        "model": "google/gemini-2.5-flash",
#        "use_system": True,
#    },
#    {
#        "model": "x-ai/grok-4.20",
#        "use_system": True,
#    }
# ]
#
#
# MODEL_SPECS = [
#    {
#        "model": "openai/gpt-5.3-chat",
#        "use_system": True,
#    },
#    {
#        "model": "anthropic/claude-opus-4.7",
#        "use_system": True,
#    },
#    {
#        "model": "google/gemini-3.5-flash",
#        "use_system": True,
#    },
#    {
#        "model": "x-ai/grok-4.3",
#        "use_system": True,
#    },
# ]
#

In [34]:
## Valid answers
VALID_ANSWERS = set("ABCDEFG")
VALID_ANSWERS

{'A', 'B', 'C', 'D', 'E', 'F', 'G'}

In [35]:
## Regex to repair (potentially) broken JSON structures
def try_repair_json(raw_text: str):
    if raw_text is None:
        return None

    text = str(raw_text).strip()

    # Remove markdown fences, e.g. ```json ... ```
    text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text).strip()

    # Extract the first JSON-looking object
    match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    if not match:
        return None

    candidate = match.group(0).strip()

    try:
        parsed = json.loads(candidate)
    except Exception:
        return None

    if not isinstance(parsed, dict):
        return None

    return parsed

In [36]:
def answer_one_from_messages(messages, model: str):
    t0 = time.time()
    usage = None

    try:
        resp = client.chat.completions.create(
            model=model,
            temperature=0,
            messages=messages,
            response_format={"type": "json_object"},
        )
        usage = getattr(resp, "usage", None)

    except Exception as e:
        latency = time.time() - t0
        return {
            "model": model,
            "answer": None,
            "explanation": None,
            "status": "error",
            "failure_reason": "api_error",
            "error_detail": str(e),
            "latency_seconds": round(latency, 3),
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
            "raw_json": None,
            "repaired_json": False,
        }

    latency = time.time() - t0

    if resp is None or not getattr(resp, "choices", None):
        return {
            "model": model,
            "answer": None,
            "explanation": None,
            "status": "error",
            "failure_reason": "no_choices_returned",
            "error_detail": None,
            "latency_seconds": round(latency, 3),
            "prompt_tokens": getattr(usage, "prompt_tokens", None),
            "completion_tokens": getattr(usage, "completion_tokens", None),
            "total_tokens": getattr(usage, "total_tokens", None),
            "raw_json": None,
            "repaired_json": False,
        }

    message = resp.choices[0].message
    raw_text = getattr(message, "content", None)

    if raw_text is None or str(raw_text).strip() == "":
        return {
            "model": model,
            "answer": None,
            "explanation": None,
            "status": "error",
            "failure_reason": "no_message_content",
            "error_detail": None,
            "latency_seconds": round(latency, 3),
            "prompt_tokens": getattr(usage, "prompt_tokens", None),
            "completion_tokens": getattr(usage, "completion_tokens", None),
            "total_tokens": getattr(usage, "total_tokens", None),
            "raw_json": raw_text,
            "repaired_json": False,
        }

    repaired_json = False

    try:
        parsed = json.loads(raw_text)
    except Exception as e:
        parsed = try_repair_json(raw_text)
        if parsed is None:
            return {
                "model": model,
                "answer": None,
                "explanation": None,
                "status": "error",
                "failure_reason": "json_parse_failed",
                "error_detail": str(e),
                "latency_seconds": round(latency, 3),
                "prompt_tokens": getattr(usage, "prompt_tokens", None),
                "completion_tokens": getattr(usage, "completion_tokens", None),
                "total_tokens": getattr(usage, "total_tokens", None),
                "raw_json": raw_text,
                "repaired_json": False,
            }
        repaired_json = True

    answer = parsed.get("answer")
    explanation = parsed.get("explanation")

    if isinstance(answer, str):
        answer = answer.strip()
    else:
        answer = str(answer).strip() if answer is not None else None

    if not isinstance(explanation, str):
        explanation = None if explanation is None else str(explanation)

    if answer not in VALID_ANSWERS:
        return {
            "model": model,
            "answer": None,
            "explanation": explanation,
            "status": "error",
            "failure_reason": "answer_not_in_valid_set",
            "error_detail": f"Returned answer: {answer}",
            "latency_seconds": round(latency, 3),
            "prompt_tokens": getattr(usage, "prompt_tokens", None),
            "completion_tokens": getattr(usage, "completion_tokens", None),
            "total_tokens": getattr(usage, "total_tokens", None),
            "raw_json": raw_text,
            "repaired_json": repaired_json,
        }

    return {
        "model": model,
        "answer": answer,
        "explanation": explanation,
        "status": "ok",
        "failure_reason": None,
        "error_detail": None,
        "latency_seconds": round(latency, 3),
        "prompt_tokens": getattr(usage, "prompt_tokens", None),
        "completion_tokens": getattr(usage, "completion_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
        "raw_json": raw_text,
        "repaired_json": repaired_json,
    }

In [37]:
## Select model
model = MODELS[0]

## Select prompt style
spec = PROMPTS[9]

## Select question
question_ = str(df_test.iloc[1]["Question"])

## Define seed
rng = np.random.default_rng(202606)

## Construct prompt/message(s) to send to model
messages = build_messages(
    question=question_,
    spec=spec,
    df_reservoir=df_reservoir,
    rng=rng,
)

## Start time
t0 = time.time()

## Grab result from model
gen_text = answer_one_from_messages(
    messages=messages,
    model=model,
)

## Stop time
t1 = time.time()

## Runtime
runtime = t1 - t0

# Print output to console
gen_text

{'model': 'deepseek/deepseek-v4-flash',
 'answer': 'E',
 'explanation': "The patient's renal function fully recovered, indicating resolved acute tubular necrosis, leaving normal kidney architecture.",
 'status': 'ok',
 'failure_reason': None,
 'error_detail': None,
 'latency_seconds': 7.235,
 'prompt_tokens': 298,
 'completion_tokens': 525,
 'total_tokens': 823,
 'raw_json': '{"answer":"E","explanation":"The patient\'s renal function fully recovered, indicating resolved acute tubular necrosis, leaving normal kidney architecture."}',
 'repaired_json': False}

In [38]:
##
## Test the 12 different prompts for a fixed model/question 
##
model = MODELS[0]
question_ = str(df_test.iloc[1]["Question"])
rng = np.random.default_rng(65432)

for spec in PROMPTS:
    messages = build_messages(
        question=question_,
        spec=spec,
        df_reservoir=df_reservoir,
        rng=rng,
    )

    t0 = time.time()
    out = answer_one_from_messages(messages=messages, model=model)
    t1 = time.time()

    print("=" * 80)
    print("PROMPT:", spec["name"])
    print("RUNTIME:", round(t1 - t0, 3), "seconds")
    print("ANSWER:", out["answer"])
    print("STATUS:", out["status"])
    print("RAW:", out["raw_json"])
    print()

PROMPT: base_zero_noex
RUNTIME: 17.784 seconds
ANSWER: E
STATUS: ok
RAW: {"answer":"E"}

PROMPT: base_one_noex
RUNTIME: 16.177 seconds
ANSWER: E
STATUS: ok
RAW: {"answer": "E"}

PROMPT: base_few_noex
RUNTIME: 10.089 seconds
ANSWER: E
STATUS: ok
RAW: {"answer": "E"}

PROMPT: base_zero_expl
RUNTIME: 18.299 seconds
ANSWER: C
STATUS: ok
RAW: {"answer":"C","explanation":"The clinical course suggests acute tubular necrosis, which heals by regeneration, leaving minimal fibrous scar."}

PROMPT: base_one_expl
RUNTIME: 2.152 seconds
ANSWER: C
STATUS: ok
RAW: {"answer":"C","explanation":"The clinical course suggests acute tubular necrosis, which heals by regeneration, leaving minimal fibrous scar."}

PROMPT: base_few_expl
RUNTIME: 10.424 seconds
ANSWER: C
STATUS: ok
RAW: {"answer":"C","explanation":"The patient had acute tubular necrosis, which heals by regeneration, not scar formation, in an autopsy after recovery."}

PROMPT: role_zero_noex
RUNTIME: 9.831 seconds
ANSWER: E
STATUS: ok
RAW: {"answ

In [39]:
##########################
## Loop over models and questions, collecting results
##########################

In [40]:
## For testing --- restrict to smaller dataset

# df_test = df_test.head(5).copy()
# df_test.shape


In [41]:
# -------------------------------------------------
# Results file setup
# -------------------------------------------------
RESULT_COLUMNS = [
    "prompt_name",
    "role",
    "explanation",
    "shots",
    "question_id",
    "question",
    "correct_response",
    "model",
    "answer",
    "status",
    "failure_reason",
    "error_detail",
    "latency_seconds",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
    "raw_json",
    "repaired_json",
]

if not os.path.exists(OUTPUT_CHECKPOINT_CSV):
    pd.DataFrame(columns=RESULT_COLUMNS).to_csv(
        OUTPUT_CHECKPOINT_CSV,
        index=False
    )


# -------------------------------------------------
# Run evaluation loop: PROMPTS x Questions
# -------------------------------------------------
results = []
t0 = time.time()

# Fix the model here
model = MODELS[0]

# Use a fixed RNG so the prompt sampling is reproducible
rng = np.random.default_rng(202606)

for spec in PROMPTS:

    print(f"\nRunning prompt: {spec['name']}")

    for i, row in df_test.iterrows():

        question_ = str(row["Question"])
        question_id_ = row["Number"] if "Number" in df_test.columns else i
        correct_response_ = str(row["CorrectResponse_R"]).strip()

        try:
            messages = build_messages(
                question=question_,
                spec=spec,
                df_reservoir=df_reservoir,
                rng=rng,
            )

            out = answer_one_from_messages(
                messages=messages,
                model=model,
            )

            result_row = {
                "prompt_name": spec["name"],
                "role": spec["role"],
                "explanation": spec["explanation"],
                "shots": spec["shots"],
                "question_id": question_id_,
                "question": question_,
                "correct_response": correct_response_,
                "model": out["model"],
                "answer": out["answer"],
                "status": out["status"],
                "failure_reason": out["failure_reason"],
                "error_detail": out["error_detail"],
                "latency_seconds": out["latency_seconds"],
                "prompt_tokens": out["prompt_tokens"],
                "completion_tokens": out["completion_tokens"],
                "total_tokens": out["total_tokens"],
                "raw_json": out["raw_json"],
                "repaired_json": out["repaired_json"],
            }

        except Exception as e:

            result_row = {
                "prompt_name": spec["name"],
                "role": spec["role"],
                "explanation": spec["explanation"],
                "shots": spec["shots"],
                "question_id": question_id_,
                "question": question_,
                "correct_response": correct_response_,
                "model": model,
                "answer": None,
                "status": "exception",
                "failure_reason": "loop_exception",
                "error_detail": str(e),
                "latency_seconds": None,
                "prompt_tokens": None,
                "completion_tokens": None,
                "total_tokens": None,
                "raw_json": None,
                "repaired_json": False,
            }

        results.append(result_row)

        pd.DataFrame([result_row]).to_csv(
            OUTPUT_CHECKPOINT_CSV,
            mode="a",
            header=False,
            index=False,
        )

        time.sleep(1.0)

t1 = time.time()
runtime = t1 - t0
print("Total runtime (seconds):", round(runtime, 3))


Running prompt: base_zero_noex

Running prompt: base_one_noex

Running prompt: base_few_noex

Running prompt: base_zero_expl

Running prompt: base_one_expl

Running prompt: base_few_expl

Running prompt: role_zero_noex

Running prompt: role_one_noex

Running prompt: role_few_noex

Running prompt: role_zero_expl

Running prompt: role_one_expl

Running prompt: role_few_expl
Total runtime (seconds): 6507.188


In [42]:
results_df = pd.DataFrame(results)
results_df.shape

(1008, 18)

In [43]:
runtime

6507.188192129135

In [44]:
runtime/60/60

1.8075522755914264

In [45]:
len(results)

1008

In [46]:
pd.DataFrame({'vars': results_df.columns})

,vars
0,prompt_name
1,role
2,explanation
3,shots
4,question_id
5,question
6,correct_response
7,model
8,answer
9,status


In [47]:
results_df.head(2)

,prompt_name,role,explanation,shots,question_id,question,correct_response,model,answer,status,failure_reason,error_detail,latency_seconds,prompt_tokens,completion_tokens,total_tokens,raw_json,repaired_json
0,base_zero_noex,False,False,0,1.1,Serum LDL-cholesterol concentrations are maeasured in blood samples collected from 25 healthy vo...,B,deepseek/deepseek-v4-flash,B,ok,NaN,NaN,7.292,225.0,235.0,460.0,"{""answer"":""B""}",False
1,base_zero_noex,False,False,0,2.1,"A 48-year-old man dies suddenly of a cardiac arrhythmia. Six weeks ago, he was resuscitated from...",E,deepseek/deepseek-v4-flash,E,ok,NaN,NaN,4.632,257.0,139.0,396.0,"{""answer"":""E""}",False


In [48]:
############
## Summarize predictions: i.e. what titles/abstracts are incl-vs-excl from review
############

In [49]:
## Determine if model response matches correct answer (from key)
results_df["is_correct"] = results_df["answer"] == results_df["correct_response"]

In [50]:
## Model
results_df.model.value_counts()

model
deepseek/deepseek-v4-flash    1008
Name: count, dtype: int64

In [51]:
##
## Inspect failure reasons
##

In [52]:
results_df["status"].value_counts(dropna=False)

status
ok       997
error     11
Name: count, dtype: int64

In [53]:
results_df["failure_reason"].value_counts(dropna=False)

failure_reason
NaN                   997
json_parse_failed       6
no_message_content      5
Name: count, dtype: int64

In [54]:
## Number of valid runs/returns for each model (i.e. non failing outputs)
n_valid_df = (
    results_df
    .groupby("model")["status"]
    .apply(lambda x: (x == "ok").sum())
    .reset_index(name="n_valid")
)

print(n_valid_df)

                        model  n_valid
0  deepseek/deepseek-v4-flash      997


In [55]:
results_df["is_valid"] = results_df["status"] == "ok"

In [56]:
#########################
## MCQ Accuracy --- by model, and stratified by model/year (2022, 2025)
#########################

In [57]:
summary_df = (
    results_df
    .groupby(["prompt_name", "role", "explanation", "shots"])
    .agg(
        n=("question_id", "size"),
        n_ok=("is_valid", "sum"),
        ok_rate=("is_valid", "mean"),
        n_correct=("is_correct", "sum"),
        accuracy=("is_correct", "mean"),
        avg_latency_seconds=("latency_seconds", "mean"),
        avg_prompt_tokens=("prompt_tokens", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        avg_total_tokens=("total_tokens", "mean"),
        repaired_rate=("repaired_json", "mean"),
    )
    .reset_index()
    .sort_values(["role", "explanation", "shots"])
)

summary_df

,prompt_name,role,explanation,shots,n,n_ok,ok_rate,n_correct,accuracy,avg_latency_seconds,avg_prompt_tokens,avg_completion_tokens,avg_total_tokens,repaired_rate
5,base_zero_noex,False,False,0,84,83,0.988095,73,0.869048,5.546345,287.819277,255.180723,543.000000,0.0
3,base_one_noex,False,False,1,84,84,1.000000,72,0.857143,4.945881,465.047619,296.976190,762.023810,0.0
1,base_few_noex,False,False,3,84,84,1.000000,72,0.857143,4.937310,789.666667,273.976190,1063.642857,0.0
4,base_zero_expl,False,True,0,84,84,1.000000,78,0.928571,4.678536,316.083333,234.214286,550.297619,0.0
2,base_one_expl,False,True,1,84,82,0.976190,73,0.869048,5.215631,486.317073,218.073171,704.390244,0.0
0,base_few_expl,False,True,3,84,84,1.000000,71,0.845238,6.922714,815.642857,318.714286,1134.357143,0.0
11,role_zero_noex,True,False,0,84,84,1.000000,78,0.928571,5.042595,303.261905,275.011905,578.273810,0.0
9,role_one_noex,True,False,1,84,83,0.988095,74,0.880952,5.237690,484.809524,273.321429,758.130952,0.0
7,role_few_noex,True,False,3,84,84,1.000000,73,0.869048,5.033500,811.916667,222.238095,1034.154762,0.0
10,role_zero_expl,True,True,0,84,84,1.000000,74,0.880952,4.786012,329.369048,222.345238,551.714286,0.0


In [58]:
summary_df.to_csv(PERFORMANCE_SUMMARY_CSV, index=False)

In [59]:
###################
## Properties of Jupyter Notebook env
####################

In [60]:
import sys
import platform
import datetime

In [61]:
## Print session info
print("=== Session Info ===")
print("Date:", datetime.datetime.now())
print("Python version:", sys.version)
print("Platform:", platform.platform())

=== Session Info ===
Date: 2026-06-04 15:49:36.383860
Python version: 3.14.3 | packaged by Anaconda, Inc. | (main, Feb 24 2026, 22:51:43) [GCC 14.3.0]
Platform: Linux-4.4.0-26100-Microsoft-x86_64-with-glibc2.31


In [63]:
## Print package info
print("=== Key Packages ===")
import openai

print("pandas:", pd.__version__)
print("openai:", openai.__version__)
print("numpy:", np.__version__)

=== Key Packages ===
pandas: 3.0.1
openai: 2.14.0
numpy: 2.4.3
